# Stage 3 — Frozen three-seed final validation run

Runs seeds 42, 43, and 44 with the approved frozen in-batch-negatives configuration. It reads only Stage 1 train/validation artifacts, never loads the test split, resumes saved checkpoints after a Colab interruption, and persists every checkpoint, embedding cache, and result under `MyDrive/RAFT`.

In [ ]:
%pip install -q "sentence-transformers==6.0.1" "transformers==5.16.1" "faiss-cpu==1.15.0" datasets pandas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import base64
import hashlib
import io
import zipfile
from pathlib import Path

MY_DRIVE = Path('/content/drive/MyDrive')
PAYLOAD_SHA256 = '6AF84F143710E46365FC1FFA9AB15AF8D0BD2F8AF294CA00A70CF6E500DA1621'
PAYLOAD_B64 = '...'

payload = base64.b64decode(PAYLOAD_B64)
assert hashlib.sha256(payload).hexdigest().upper() == PAYLOAD_SHA256
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    archive.extractall(MY_DRIVE)
print('Frozen Stage 3 source and configs restored to Drive.')

In [ ]:
import json
import os
import subprocess
import sys

DRIVE_ROOT = Path('/content/drive/MyDrive/RAFT')
REPO_DIR = DRIVE_ROOT / 'stage3_inbatch' / 'final_source_snapshot'
DATA_DIR = DRIVE_ROOT / 'stage1_scifact' / 'processed'
BASELINE = DRIVE_ROOT / 'stage2_baselines' / 'results' / 'stage2_baselines.csv'
SMOKE = DRIVE_ROOT / 'stage3_inbatch' / 'smoke_seed42' / 'results' / 'stage3_smoke_result.json'
os.environ['HF_HOME'] = str(DRIVE_ROOT / 'huggingface_cache')

for required in [DATA_DIR / 'corpus.jsonl', DATA_DIR / 'queries_train.jsonl', DATA_DIR / 'qrels_train.tsv', DATA_DIR / 'queries_validation.jsonl', DATA_DIR / 'qrels_validation.tsv', BASELINE, SMOKE]:
    assert required.is_file(), f'Missing required Drive artifact: {required}'
for seed in (42, 43, 44):
    subprocess.run([sys.executable, str(REPO_DIR / 'src' / 'train.py'), '--mode', 'final', '--config', str(REPO_DIR / 'configs' / f'stage3_inbatch_final_seed{seed}.json'), '--data-dir', str(DATA_DIR), '--baseline-results', str(BASELINE), '--output-dir', str(DRIVE_ROOT / 'stage3_inbatch' / f'final_seed{seed}'), '--validate-only'], check=True)
print('All three frozen configs validated; test split was not loaded.')

In [ ]:
subprocess.run([sys.executable, str(REPO_DIR / 'src' / 'run_stage3_final.py'), '--repo-dir', str(REPO_DIR), '--drive-root', str(DRIVE_ROOT)], check=True)

In [ ]:
import pandas as pd

RESULTS_DIR = DRIVE_ROOT / 'stage3_inbatch' / 'final_results'
SUMMARY = RESULTS_DIR / 'stage3_final_summary.json'
summary = json.loads(SUMMARY.read_text())
assert summary['test_split_loaded'] is False
display(pd.read_csv(RESULTS_DIR / 'stage3_final_per_seed_validation.csv'))
display(pd.read_csv(RESULTS_DIR / 'stage3_final_aggregate_validation.csv'))
print('Attach this file in the Codex task:', SUMMARY)